In [1]:
pip install faster-whisper sounddevice scipy pyttsx3 transformers torch


Note: you may need to restart the kernel to use updated packages.


In [2]:
import sounddevice as sd
from scipy.io.wavfile import write
from faster_whisper import WhisperModel
import pyttsx3
from transformers import pipeline


In [3]:
# Speech to Text
whisper_model = WhisperModel(
    "base",
    device="cpu",
    compute_type="int8"
)

# Question Answering Model
chatbot = pipeline(
    "text-generation",
    model="microsoft/DialoGPT-small"
)

# Text to Speech
speaker = pyttsx3.init()


# Question Answering Model
chatbot = pipeline("text-generation", model="microsoft/DialoGPT-small", clean_up_tokenization_spaces=False)

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

In [4]:
def record_audio():
    fs = 16000
    duration = 10

    print("🎤 Speak now...")

    audio = sd.rec(
        int(duration * fs),
        samplerate=fs,
        channels=1
    )

    sd.wait()

    write("recorded.wav", fs, audio)

    print("✅ Recording Finished")

In [5]:
def speech_to_text():
    
    segments, info = whisper_model.transcribe(
        "recorded.wav"
    )

    text = ""

    for segment in segments:
        text += segment.text

    return text.strip()

In [6]:
def get_answer(question):

    prompt = f"Question: {question}\nAnswer:"

    response = chatbot(
        prompt,
        max_new_tokens=50,
        do_sample=True
    )

    answer = response[0]["generated_text"]

    return answer

In [7]:
def speak(text):

    speaker.say(text)
    speaker.runAndWait()

In [8]:
record_audio()

user_text = speech_to_text()

print("🧑 You:", user_text)

answer = get_answer(user_text)

print("🤖 Assistant:", answer)

speak(answer)


🎤 Speak now...
✅ Recording Finished
🧑 You: Thank you very much.
🤖 Assistant: Question: Thank you very much.
Answer: Yes


In [9]:
user_text = speech_to_text()

if len(user_text.strip()) < 3:
    print("⚠️ No speech detected. Please speak again.")
else:
    answer = get_answer(user_text)
    print("🤖 Assistant:", answer)
    speak(answer)
    

🤖 Assistant: Question: Thank you very much.
Answer: Yes .
